# Week 3 Lab. Markov Chains and the Stationary Distribution
**Time Series Analysis & Random Processes** · Graduate School of Data Science, Chonnam National University · notebook v5 (2026-09-15, TODOs moved onto the Markov concepts)

---

### What this lab does

Last week asked when one long record can reveal a process. Today we answer that question with a finite-state Markov model.

Record Sunny as 1 and every other state as 0. The average of these numbers is exactly the sunny visit frequency.
Counting visits in this lab is last week's time average in a concrete form.

Stationarity gives stable quantities to study. The ACF describes linear dependence across time; it is not a third assumption to pass.
Time-average convergence explains when one long simulated path estimates the model's stationary probabilities.
We also distinguish that convergence from convergence of the state probabilities at a particular time.

> 요약: 맑음이면 1, 나머지는 0으로 기록하면 평균이 곧 맑은 날의 비율입니다. 방문 비율을 세는 이번 실습이 지난주의 시간평균입니다. 정상성·ACF·에르고딕성을 모두 통과해야만 시작하는 실습은 아닙니다.

| Step | What you do |
|---|---|
| 1 | Write down a transition matrix and step a distribution forward with `mu @ P` |
| 2 | Simulate the chain by hand (Monte Carlo) and count visits |
| 3 | Use the supplied linear solve and verify `πP = π` |
| 4 | Compare the linear solve and `P^n`; allow sampling error in Monte Carlo |
| 5 | Watch `P^n` converge, and measure how fast |
| 6 | Break aperiodicity, then break irreducibility, and see which method still works |
| 7 | PageRank on a four-page web, with and without teleportation |

### How to use it

Two cells are marked **`TODO`**. Write those yourself first; they are the part worth doing by hand.
직접 채워 보는 것이 이 노트북의 핵심입니다.

> Each `TODO` is one line that reads `… = None`. Replace the `None` with your code; there is nothing to delete.
> If you run the cell before that, it stops with a `NotImplementedError` whose message says exactly this.
> **That is expected, not a broken notebook.** Each `TODO` is followed by a collapsed **Solution (정답)** cell:
> run it and the notebook continues from there. Open it only after you have tried.
> 각 `TODO` 는 `… = None` 한 줄입니다. `None` 자리에 코드를 넣으면 되고, 지울 줄은 없습니다.

> ### Before you touch anything: **File > Save a copy in Drive**
> The link I posted opens **read-only**. Colab will say *"changes will not be saved"*.
> Save your own copy first, or everything you type here disappears when you close the tab.
>
> **Nothing to submit this week.** Assignment 1 (12%) goes out after Week 4 and will use exactly this machinery,
> so the two TODO cells are the ones to get right now.

---
## 0. Setup

Nothing to install this week. Everything below ships with Colab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

SEED = 42                       # fixed seed = reproducible; every graded submission needs one
rng  = np.random.default_rng(SEED)

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (11, 3.2)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# 빈칸(TODO)을 아직 안 채웠을 때 보여 줄 안내입니다. 고장이 아니라는 뜻입니다.
BLANK_MSG = (
    "\n\n"
    "  정상입니다. 고장이 아니라 일부러 비워 둔 칸입니다.\n"
    "     위의 None 자리에 코드를 넣고 이 셀을 다시 실행하세요. 지울 줄은 없습니다.\n"
    "     막힐 때 바로 아래 'Solution / 정답' 셀을 실행하면 이어서 진행됩니다.\n\n"
    "  This is expected, not a broken notebook.\n"
    "     Replace None above with your code and re-run. Nothing needs deleting.\n"
    "     If stuck, run the Solution cell just below and continue.\n"
)

print("numpy:", np.__version__)

### Optional. Korean labels in plots / 그림에 한글 쓰기

Every figure here is labelled in English, so you can skip this. But Colab ships **no Korean font**, so the moment you
write a Korean title yourself the characters come out as boxes. Run the cell below and it is fixed for the rest of
the session, **no runtime restart needed**.

이 노트북의 그림은 전부 영문이라 건너뛰어도 됩니다. 다만 Colab에는 한글 폰트가 없어서, 여러분이 한글 제목을 쓰는
순간 네모로 깨집니다. 아래 셀을 실행하면 **런타임 재시작 없이** 바로 해결됩니다.

In [ ]:
#@title ▶ 한글 폰트 설치 / Install Korean font (optional) { display-mode: "form" }
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1
import matplotlib.font_manager as fm

_p = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
try:
    fm.fontManager.addfont(_p)                  # 캐시 재생성 없이 즉시 등록 -> 재시작 불필요
    plt.rc("font", family=fm.FontProperties(fname=_p).get_name())
    plt.rc("axes", unicode_minus=False)         # 음수 눈금 깨짐 방지
    print("Korean font ready:", plt.rcParams["font.family"][0])
except Exception as _e:
    # apt 가 막혀도 노트북 전체가 멈추지 않도록 한다. 그림 라벨만 영문/네모로 나온다.
    plt.rc("axes", unicode_minus=False)
    print("한글 폰트를 건너뜁니다 (그림 라벨이 깨질 수 있습니다):", _e)

# 함정 1. 라벨에 유니코드 마이너스(U+2212)나 공집합(U+2205)을 직접 타이핑하면
#         이 폰트에 글리프가 없어 네모로 뜹니다. ASCII 하이픈(-)을 쓰세요.
# 함정 2. 로그 축 눈금(10^-3 등)은 mathtext 로 그려지므로 unicode_minus=False 로도 막히지 않습니다.
#         로그 축을 쓸 때는 FuncFormatter 로 눈금 문자열을 직접 만드세요:
#           from matplotlib.ticker import FuncFormatter
#           ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"1e{int(round(np.log10(y)))}"))


---
## 1. A chain, and one tick of the clock

Three weather states, **Sunny, Cloudy, Rainy**. Row *i* of `P` is the full conditional distribution of tomorrow
given that today is state *i*, so **every row must sum to 1** (deck: “The Transition Matrix P”).

The only dynamics you need all week: if `mu` is today's distribution as a row vector, tomorrow's is `mu @ P`.

### Same idea, a different weather model
The lecture's two-state example uses Sunny and Rainy and has sunny stationary probability two-thirds.
This lab uses **three** states and a different transition matrix. Its stationary probabilities are **21/46, 13/46, 12/46**.
Do not copy the lecture's two-thirds into this calculation.

We keep P fixed over time (time homogeneity). Starting with `mu = [1, 0, 0]` does not make the process stationary:
the first day's sunny probability is 1 and the next day's is 0.7. Starting instead from the stationary distribution would preserve the distribution at every step.

> 요약: 덱은 두 상태, 실습은 세 상태이므로 숫자는 다릅니다. 전이표가 일정한 시간동질성과 분포가 유지되는 정상성도 구분하세요. 덱 「One Weather Record, Two Weeks of Questions」·「The Weather Example Answers Last Week」.

In [ ]:
STATES = ["Sunny", "Cloudy", "Rainy"]

P = np.array([[0.70, 0.20, 0.10],      # from Sunny
              [0.30, 0.40, 0.30],      # from Cloudy
              [0.20, 0.30, 0.50]])     # from Rainy

print("row sums:", P.sum(axis=1))      # must be 1, 1, 1
assert np.allclose(P.sum(axis=1), 1), "rows must sum to 1; that is what 'stochastic matrix' means"

mu = np.array([1.0, 0.0, 0.0])         # certain it is Sunny today
print("\nday  " + "  ".join(f"{s:>7}" for s in STATES))
for day in range(6):
    print(f"{day:3d}  " + "  ".join(f"{p:7.4f}" for p in mu))
    mu = mu @ P

The distribution is approaching a fixed row. For this finite, irreducible, aperiodic chain, every initial distribution approaches the same stationary distribution.

That is one question. Another is whether visit frequencies along one long path approach that distribution.
The alternator later in the lab separates these two questions: its time averages converge even though its state probabilities can keep oscillating.

> 요약: 시각별 상태확률의 수렴과 한 경로의 방문 비율 수렴은 서로 다른 주장입니다. 뒤의 교대 연쇄에서 차이를 봅니다.

---
## 2. Route ①, simulate the chain

The most literal reading of the model: stand in a state, roll a die weighted by that row, move, repeat.

### The time average hidden in the counts
For each visited state, imagine writing 1 if it is Sunny and 0 otherwise.
The sum of these indicators is `counts[0]`; their mean is `counts[0] / n_steps`.
Repeat for Cloudy and Rainy to obtain all three visit frequencies.

For a finite irreducible chain, these frequencies approach the unique stationary distribution, even from a fixed starting state.
Aperiodicity is not required for this time-average conclusion.

> 요약: `counts[0] / n_steps`는 맑음 지시변수의 표본평균입니다. 유한 기약 연쇄의 장기 방문 비율 정리가 이 평균을 정당화합니다. 덱 「Three Different Questions, One Similar Answer」.

### 이 노트북에서 쓰는 NumPy

파이썬이 처음이어도 따라올 수 있도록, 이 노트북에 나오는 것만 모아 둡니다. 외울 필요는 없고 막힐 때 돌아와 보면 됩니다.

| 쓰는 것 | 뜻 |
|---|---|
| `len(M)` | 행렬 `M` 의 행 수. 전이행렬은 n×n 이므로 이것이 곧 상태 개수 |
| `M[0]`, `M[1]` | 행렬 `M` 의 첫 번째 행, 두 번째 행 |
| `M[0, 2]` | 0행 2열의 값 하나 |
| `np.zeros(n)` | 길이 n 인 0 벡터 |
| `v @ M` | 행벡터 `v` 와 행렬 `M` 의 곱 |
| `np.allclose(a, b)` | 두 배열이 반올림 오차 범위에서 같으면 `True` |
| `np.linalg.matrix_power(M, n)` | `M` 의 n 제곱 |
| `np.random.default_rng(SEED)` | 난수 생성기 `rng` 를 만든다. 같은 `SEED` 를 쓰면 같은 난수열을 재현할 수 있다 |

**`rng.choice(n, p=q)`**

`0, 1, ..., n-1` 중 하나를 확률 벡터 `q` 에 따라 무작위로 고릅니다. `q` 의 길이는 `n` 이고 합은 1 이어야 합니다.

```python
rng.choice(2, p=[0.8, 0.2])
# 0 을 0.8 의 확률로, 1 을 0.2 의 확률로 돌려준다
```

날씨로 옮기면, 내일 맑음이 0.8 비가 0.2 일 때 위 한 줄이 곧 내일 날씨를 한 번 뽑는 것입니다.

아래 TODO 에서 생각할 것은 하나입니다. **지금 상태가 `s` 일 때, 다음 상태로 갈 확률은 `P` 의 어느 행에 있는가.**


In [ ]:
# ---------------------------------------------------------------- TODO 1 ----
# 다음 이름을 씁니다
#   P       : 전이행렬. P[i, j] = 상태 i 에서 상태 j 로 갈 확률
#   n       : 상태 개수
#   s       : 현재 상태 번호. 0 = Sunny, 1 = Cloudy, 2 = Rainy
#   probs   : 지금 상태에서 다음 상태로 갈 확률 벡터   <- 여기가 빈칸
#   counts  : 상태별 방문 횟수를 담는 배열
#   rng     : 난수 생성기
#
# 참고  rng.choice(n, p=q)
#   0, 1, ..., n-1 중 하나를 확률 벡터 q 에 따라 무작위로 고른다.
#   예)  rng.choice(2, p=[0.8, 0.2])  ->  0 을 0.8, 1 을 0.2 의 확률로 돌려준다.

def simulate_chain(P, n_steps, rng, start=0):
    """Run the chain for n_steps and return the visit counts as an array of length len(P).

    counts[j] = how many of the n_steps visits landed in state j.
    """
    n = len(P)             # 상태 개수. P 가 n x n 이므로 행 수가 곧 상태 수
    counts = np.zeros(n)   # 상태별 방문 횟수를 0 으로 초기화
    s = start              # 현재 상태를 시작 상태로 둔다

    for _ in range(n_steps):
        # TODO 1. 지금 상태가 s 일 때, 다음 상태로 갈 확률은 P 의 어느 행에 있나요?
        #   그 행을 probs 에 넣으세요. 한 줄입니다.
        probs = None
        if probs is None:
            raise NotImplementedError(BLANK_MSG)

        s = rng.choice(n, p=probs)   # 그 확률에 따라 다음 상태를 하나 뽑는다
        counts[s] += 1               # 방금 도착한 상태의 방문 횟수 +1

    return counts
# --------------------------------------------------------------------------


counts = simulate_chain(P, 200_000, np.random.default_rng(SEED), start=0)
pi_mc = counts / counts.sum()
print("Monte Carlo frequencies:", pi_mc)
# 기대값  Monte Carlo frequencies: [0.4578 0.2816 0.2606]
#   시드를 고정했으므로 이 값이 그대로 나와야 합니다.
#   [0.3333 0.3333 0.3333] 근처가 나왔다면 p= 를 안 쓰고 균등 추출이 된 것입니다.


In [ ]:
#@title ▶ Solution / 정답. Run only if you are stuck (this overwrites your version) { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.

def simulate_chain(P, n_steps, rng, start=0):
    """Run the chain for n_steps and return the visit counts as an array of length len(P).

    counts[j] = how many of the n_steps visits landed in state j.
    """
    n = len(P)             # 상태 개수. P 가 n x n 이므로 행 수가 곧 상태 수
    counts = np.zeros(n)   # 상태별 방문 횟수를 0 으로 초기화
    s = start              # 현재 상태를 시작 상태로 둔다

    for _ in range(n_steps):
        probs = P[s]                 # 빈칸이었던 줄. 지금 상태 s 의 행이 곧 확률 벡터다
        if probs is None:
            raise NotImplementedError(BLANK_MSG)

        s = rng.choice(n, p=probs)   # 그 확률에 따라 다음 상태를 하나 뽑는다
        counts[s] += 1               # 방금 도착한 상태의 방문 횟수 +1

    return counts


counts = simulate_chain(P, 200_000, np.random.default_rng(SEED), start=0)
pi_mc = counts / counts.sum()
print("Monte Carlo frequencies:", pi_mc)
# 기대값  Monte Carlo frequencies: [0.4578 0.2816 0.2606]
#   시드를 고정했으므로 이 값이 그대로 나와야 합니다.
#   [0.3333 0.3333 0.3333] 근처가 나왔다면 p= 를 안 쓰고 균등 추출이 된 것입니다.


---
## 3. Route ②, solve πP = π

The deck's “The Stationary Distribution” defines π as the row vector with `πP = π` and `Σπ = 1`. Transposing turns the first part into the familiar
`Aπᵀ = 0` shape:

$$\pi P = \pi \iff (P^{\mathsf T} - I)\,\pi^{\mathsf T} = 0$$

Three equations, but only two of them are independent (each row of `P − I` sums to 0, so the rows of `Pᵀ − I` add up to the zero row), so the normalization
`Σπ = 1` is not an extra decoration; it is the equation that pins the answer down. Stack it on and solve the
4×3 least-squares system.

### 손으로 따라가 보기, 2상태 예

아래 셀의 `A` 와 `b` 는 통째로 주어집니다. 무슨 일을 하는 코드인지는 2상태 예로 한 번만 보면 충분합니다.

$$P = \begin{pmatrix} 0.8 & 0.2 \\ 0.4 & 0.6 \end{pmatrix}, \qquad \pi = (\pi_S,\ \pi_R)$$

에서 `πP = π` 는 성분별로 이렇습니다.

$$0.8\,\pi_S + 0.4\,\pi_R = \pi_S$$
$$0.2\,\pi_S + 0.6\,\pi_R = \pi_R$$

여기에 정규화 한 줄을 더합니다.

$$\pi_S + \pi_R = 1$$

**아래 코드가 하는 일은 이 연립방정식을 세워 컴퓨터에 넘기는 것, 그게 전부입니다.** 각 조각의 역할은 이렇습니다.

| 코드 | 위 식에서 |
|---|---|
| `P.T` | 좌변을 정리할 때 쓰는 P 의 전치 |
| `np.eye(n)` | n×n 단위행렬 I. `P.T - I` 가 위 두 식의 좌변 |
| `np.ones(n)` | `[1, 1, ...]`, 즉 `πS + πR` 을 만드는 줄 |
| `np.zeros(n)` | 위 두 식의 우변인 `[0, 0]` |
| `np.vstack` | 식들을 세로로 쌓아 하나의 행렬 `A` 로 |
| `np.concatenate` | 우변들을 이어 붙여 하나의 벡터 `b` 로 |
| `np.linalg.lstsq(A, b, rcond=None)[0]` | `A @ x = b` 를 푸는 NumPy 함수. `[0]` 이 해 벡터 |

이번 주에 맞혀야 할 것은 `lstsq` 의 철자가 아닙니다. **구한 π 가 정말 정상분포인지 확인하는 것**입니다. 그게 아래 TODO 2 입니다.


In [ ]:
# ---------------------------------------------------------------- TODO 2 ----
# 다음 이름을 씁니다
#   P         : 전이행렬 (n x n)
#   A, b      : 위 표에서 설명한 선형계. 아래에서 만들어 둡니다
#   pi_solve  : 풀어서 얻은 정상분포 후보
#   check     : 그것이 정말 정상분포인지의 판정   <- 여기가 빈칸
#
# 참고  np.allclose(a, b)
#   두 배열이 반올림 오차 범위에서 같으면 True 를 돌려준다.
#   예)  np.allclose([1.0, 2.0], [1.0, 2.0000000001])  ->  True

def stationary_by_solve(P):
    """Return the stationary distribution as a 1-D array, by solving the linear system."""
    n = len(P)

    # πP = π 를 열벡터로 옮기면 (P.T - I) π = 0 이다. 이것이 위 n 줄.
    # 마지막 한 줄 np.ones(n) 은 π 의 성분을 모두 더하는 줄이고, 우변의 1 과 짝지어 Σπ = 1 을 건다.
    A = np.vstack([P.T - np.eye(n), np.ones(n)])   # (n+1) x n

    # 위 n 개 식의 우변은 0, 마지막 정규화식의 우변은 1.
    b = np.concatenate([np.zeros(n), [1.0]])       # 길이 n+1

    return np.linalg.lstsq(A, b, rcond=None)[0]    # [0] 이 해 벡터다


pi_solve = stationary_by_solve(P)
print("pi (linear solve):", pi_solve)

# TODO 2. 정상분포의 정의는 "한 번 더 이동시켜도 그대로"입니다.
#   위에서 구한 pi_solve 를 한 번 더 이동시킨 것과, 이동시키기 전의 것.
#   이 둘이 같은지 np.allclose 로 확인해 check 에 넣으세요. 한 줄입니다.
#   (한 번 이동시키는 것은 행벡터에 P 를 오른쪽에서 곱하는 것입니다.)
check = None
if check is None:
    raise NotImplementedError(BLANK_MSG)

print("check pi @ P == pi:", check)
# --------------------------------------------------------------------------
# 기대값  pi (linear solve): [0.4565 0.2826 0.2609]   ·   check pi @ P == pi: True
#   §2 의 몬테카를로 값과 소수 셋째 자리에서 갈리는 것이 정상입니다.
#   check 가 False 라면 pi_solve @ P 가 아니라 P @ pi_solve 를 썼을 가능성이 큽니다.
#   이 노트북의 P 는 행이 출발 상태이므로 분포는 왼쪽에서 곱합니다.


In [ ]:
#@title ▶ Solution / 정답. Run only if you are stuck { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.

def stationary_by_solve(P):
    """Return the stationary distribution as a 1-D array, by solving the linear system."""
    n = len(P)

    # πP = π 를 열벡터로 옮기면 (P.T - I) π = 0 이다. 이것이 위 n 줄.
    # 마지막 한 줄 np.ones(n) 은 π 의 성분을 모두 더하는 줄이고, 우변의 1 과 짝지어 Σπ = 1 을 건다.
    A = np.vstack([P.T - np.eye(n), np.ones(n)])   # (n+1) x n

    # 위 n 개 식의 우변은 0, 마지막 정규화식의 우변은 1.
    b = np.concatenate([np.zeros(n), [1.0]])       # 길이 n+1

    return np.linalg.lstsq(A, b, rcond=None)[0]    # [0] 이 해 벡터다


pi_solve = stationary_by_solve(P)
print("pi (linear solve):", pi_solve)

check = np.allclose(pi_solve @ P, pi_solve)   # 빈칸이었던 줄. 한 번 이동시켜도 그대로인지
if check is None:
    raise NotImplementedError(BLANK_MSG)

print("check pi @ P == pi:", check)
# 기대값  pi (linear solve): [0.4565 0.2826 0.2609]   ·   check pi @ P == pi: True
#   §2 의 몬테카를로 값과 소수 셋째 자리에서 갈리는 것이 정상입니다.
#   check 가 False 라면 pi_solve @ P 가 아니라 P @ pi_solve 를 썼을 가능성이 큽니다.
#   이 노트북의 P 는 행이 출발 상태이므로 분포는 왼쪽에서 곱합니다.


---
## 4. Three different questions, one similar answer

세 방법이 비슷한 숫자를 냅니다. 숫자를 보기 전에, **각각이 무엇을 계산하는 방식인지** 먼저 구분해 둡니다. 같은 것의 세 문법이 아니라 서로 다른 개념입니다.

| | 묻는 것 | 얻는 것 |
|---|---|---|
| **정상방정식** `πP = π` | 한 번 더 이동시켜도 안 변하는 분포가 있는가 | 그 분포를 직접 구한다 |
| **행렬 거듭제곱** `P⁵⁰` | 50 스텝 뒤의 상태확률은 어떻게 되는가 | 수렴하는 연쇄라면 각 행이 π 에 가까워진다 |
| **한 번의 긴 시뮬레이션** | 한 경로를 오래 걸으면 각 상태에 몇 퍼센트 머무는가 | 방문 비율. 시간평균이다 |

세 번째는 앞의 둘과 성격이 다릅니다. 앞의 둘은 **분포**에 대한 계산이고, 마지막은 **경로 하나**에서 센 것입니다. 이번 주 덱 「Three Different Questions, One Similar Answer」과 2주차의 시간평균 이야기가 만나는 자리입니다.

참고로 이 날씨 연쇄는 해석적으로도 풀립니다. 정확한 답은 분수로 `21/46, 13/46, 12/46` 입니다. 아래에서는 이 값을 기준으로 세 방법이 각각 얼마나 떨어져 있는지 봅니다.


In [ ]:
P50 = np.linalg.matrix_power(P, 50)
pi_power = P50[0]   # 이 예제에서는 n=50 이면 어느 행을 골라도 수치적으로 거의 같다

print("P^50 =\n", P50)
print("\nrows of P^50 identical to 4 decimals:", np.allclose(P50, P50[0], atol=1e-4))

# 해석해. 손으로 연립방정식을 풀면 나오는 분수 값이다.
pi_exact = np.array([21/46, 13/46, 12/46])

print(f"\n{'':>14}{'Sunny':>9}{'Cloudy':>9}{'Rainy':>9}")
for name, v in [("exact (분수)", pi_exact),
                ("linear solve", pi_solve),
                ("P^50 row", pi_power),
                ("Monte Carlo", pi_mc)]:
    print(f"{name:>14}" + "".join(f"{x:9.4f}" for x in v))

print("\n해석해와의 차이, 그리고 그 차이가 어디서 오는지")
print(f"  linear solve : {np.max(np.abs(pi_solve - pi_exact)):.2e}   부동소수 반올림")
print(f"  P^5  row     : {np.max(np.abs(np.linalg.matrix_power(P, 5)[0] - pi_exact)):.2e}   극한 근사 오차. n 이 작으면 이만큼 남는다")
print(f"  P^50 row     : {np.max(np.abs(pi_power - pi_exact)):.2e}   같은 방법인데 n=50 이면 반올림 수준까지 내려왔다")
print(f"  Monte Carlo  : {np.max(np.abs(pi_mc   - pi_exact)):.5f}   20만 번 추출의 표본오차")


세 줄의 오차가 **서로 다른 이유**로 생깁니다. 크기만 보지 말고 출처를 보세요.

* **linear solve** 는 부동소수 반올림뿐이라 10⁻¹⁶ 수준입니다. 사실상 해석해와 같다고 봐도 됩니다.
* **Pⁿ 의 한 행** 은 유한한 n 에서의 n 스텝 분포이므로, 정상해와 정확히 같다고 일반적으로 보장되지 않습니다. 이 예제에서는 n=5 일 때 1.2e-02 의 차이가 남고, n=50 에서는 반올림 수준까지 작아집니다. 값이 작아졌을 뿐 방법의 성격이 바뀐 것은 아니므로 이것을 "정확한 방법"이라고 부르지 않습니다.
* **Monte Carlo** 는 표본추정입니다. 20만 스텝에서 상태당 표준오차가 약 0.002 입니다(스텝이 서로 상관되어 있어 독립 추출의 0.001 보다 큽니다). 고정 시드에서는 0.0013 이 나오고, 다른 시드면 0.003 이나 0.004 도 흔합니다. 20만 스텝이어도 실현된 오차가 반드시 0.01 아래라는 보장은 없습니다. 0.01 을 넘으면 곧바로 버그라고 단정하지 말고, 시드와 표본오차, 구현을 함께 점검할 **경고 신호**로 읽습니다.

한 방법이 정말 틀렸을 때 가장 흔한 원인은 행렬을 전치해 쓴 것입니다(덱 「Common Pitfalls」). 이 노트북의 `P` 는 행이 출발 상태인 row-stochastic 행렬이고, 분포는 **왼쪽에서** 곱합니다. `mu @ P`.

π 읽기: 장기적으로 맑음 약 46%, 흐림 28%, 비 26% 이고, 이는 오늘 날씨와 무관합니다.


---
## 5. How fast is "eventually"?

`Pⁿ → π` 는 극한입니다. 그러니 실제로 몇 스텝이면 쓸 만해지는지가 궁금해집니다. 먼저 `Pⁿ` 을 몇 개 찍어 놓고 **눈으로** 봅니다. 세 행이 서로 닮아 가고, 닮아 간 그 행이 π 입니다.


In [ ]:
# 먼저 눈으로. n 이 커질수록 세 행이 서로 닮아 갑니다.
for n in [1, 2, 5, 10, 20]:
    print(f"P^{n} =")
    print(np.linalg.matrix_power(P, n))
    print()

# 같은 것을 한 장으로. 남은 오차의 최댓값을 로그 눈금에 찍습니다.
ns = np.arange(1, 31)
err = [np.abs(np.linalg.matrix_power(P, n) - pi_solve).max() for n in ns]

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.semilogy(ns, err, "o-", ms=4, color="#3b6ea5")
# Log ticks are drawn as 10^-3 via mathtext, whose minus sign is missing from the Korean font.
# Formatting them as plain text keeps the axis readable whether or not you ran the font cell.
ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"1e{int(round(np.log10(y)))}"))
ax.set_xlabel("n"); ax.set_ylabel("max |P^n - pi|")
ax.set_title("Convergence of P^n to the stationary distribution")
plt.tight_layout(); plt.show()

for n in [1, 5, 10, 20]:
    print(f"  n = {n:2d}   max error = {err[n-1]:.2e}")


# ── Optional. For the curious ─────────────────────────────────────────────
# 이번 주 범위는 아닙니다. 궁금한 사람만 보세요.
# 수렴 속도는 P 의 두 번째로 큰 고유값의 크기와 관련이 있습니다.
# 매 스텝 남은 오차에 대략 그 값이 곱해지므로, 로그 눈금에서 직선으로 보입니다.
eig = np.sort(np.abs(np.linalg.eigvals(P)))[::-1]
print("\n[Optional] eigenvalues of P by size:", eig)
print("[Optional] |lambda_2| =", round(eig[1], 4))


로그 눈금에서 직선이면 오차가 기하급수적으로 줄고 있다는 뜻입니다. 여기서는 열 스텝이면 이미 0.001 아래입니다. 이번 주에 가져갈 것은 이 한 문장입니다. **`Pⁿ` 은 n 이 커질수록 각 행이 π 로 다가간다.**

> Optional / For the curious. 이 직선의 기울기는 P 의 두 번째로 큰 고유값 크기(여기서는 약 0.47)가 정합니다. 고유값 1 은 π 자신의 몫이고 나머지가 감쇠합니다. 6주차 이후에 다시 만납니다.

### Where last week's ACF fits
Encode Sunny as 1 and the other states as 0. In the stationary version of this finite irreducible aperiodic chain,
the ACF of that indicator fades with the lag. It describes the echo that affects the precision of its sample mean.
It is not another assumption that must be checked before running a Markov chain.

The lecture's **two-state** weather indicator has ACF `0.4**h` in stationarity. That formula does not automatically apply to this **three-state** matrix.
The matrix-convergence error plotted above is not itself an ACF plot.

> 요약: ACF는 이 실습에서도 맑음 여부의 시간 의존을 설명합니다. 위 그래프는 ACF가 아니라 행렬 수렴 오차입니다. 덱의 두 상태 ACF를 세 상태 모형에 그대로 쓰지 마세요.


---
## 6. Experiment A, break aperiodicity

The strict alternator from the deck's “Discussion. Why Aperiodicity?”: A → B → A → B, never staying put. Nothing here is random.

아래 셀은 먼저 `Pⁿ` 만 찍습니다. 숫자를 보고 한 가지만 답해 보세요. **`Pⁿ` 이 한 곳으로 가라앉나요?** 그다음에 앞에서 만든 두 함수를 그대로 가져다 씁니다. 새로 구현할 것은 없습니다.


In [ ]:
P_alt = np.array([[0.0, 1.0],
                  [1.0, 0.0]])       # A -> B -> A -> B. 제자리에 머무는 일이 없다

# 먼저 이것만 봅니다. P^n 이 한 곳으로 가라앉나요?
print("P^n for the alternator:")
for n in range(1, 7):
    print(f"  n = {n}:", np.linalg.matrix_power(P_alt, n).ravel())

# 이제 §2·§3 에서 만든 함수를 그대로 가져다 씁니다.
print("\npi from the linear solve:", stationary_by_solve(P_alt))
print("pi from Monte Carlo    :", (lambda c: c / c.sum())(
      simulate_chain(P_alt, 100_000, np.random.default_rng(1), start=0)))
# 기대값  P^n 은 단위행렬과 교환행렬을 영원히 왕복합니다.
#         정상분포는 [0.5 0.5], 몬테카를로도 [0.5 0.5] 근처입니다.


Read the three answers carefully. They do **not** say the same thing.

- `P^n` flips between the identity and the swap forever. It never converges, exactly as the deck's “Discussion. Why Aperiodicity?” claims.
- The linear solve still returns (0.5, 0.5): a stationary distribution **exists**. Start from (0.5, 0.5) and you
  stay at (0.5, 0.5). The fixed point is real; it is just never reached from a corner.
- Monte Carlo also returns (0.5, 0.5), because the *time average* over a long path does converge even here.

So periodicity kills one of the three routes, not all three. "Existence" and "convergence of `P^n`" are different
claims about different objects (the same deck slide), and the time-average is a third object again.

### Why this does not contradict Week 2
If we choose the alternator's initial state with equal probabilities, it is stationary.
Its 0/1 indicator alternates, so its ACF does not fade, yet its average approaches one-half.
Last week's decaying-autocovariance condition was **sufficient**, not necessary, for mean-square convergence of the sample mean of a weakly stationary process.

> 요약: 메아리가 사라지면 평균 수렴을 보일 수 있지만, 평균 수렴에 반드시 메아리가 사라져야 하는 것은 아닙니다. 덱 「Discussion. Why Aperiodicity?」·「When Does P^n Actually Converge?」.

---
## 7. Experiment B, break irreducibility

Two closed groups that never talk to each other: states 0–1 form one world, states 2–3 another.

아래 셀도 숫자를 내기 전에 행렬만 보고 답해 보세요. **0-1 세계에서 2-3 세계로 건너갈 수 있나요?** 갈 수 없다면, 출발한 세계 밖으로 나갈 수 없다는 뜻이고, 그러면 모두에게 통하는 하나의 장기 분포를 기대할 수 없습니다.


In [ ]:
P_split = np.array([[0.5, 0.5, 0.0, 0.0],    # 0 -> 0 또는 1
                    [0.5, 0.5, 0.0, 0.0],    # 1 -> 0 또는 1
                    [0.0, 0.0, 0.2, 0.8],    # 2 -> 2 또는 3
                    [0.0, 0.0, 0.8, 0.2]])   # 3 -> 2 또는 3

# 먼저 행렬만 봅니다. 건너갈 수 있나요?
print("0 에서 2 나 3 으로 갈 확률:", P_split[0, 2:])
print("2 에서 0 이나 1 로 갈 확률:", P_split[2, :2])

# 시작 상태를 바꿔 가며 §2 의 함수를 그대로 돌립니다. 답이 달라집니다.
print("\nMonte Carlo, 시작 상태만 바꿔서")
for start in [0, 2]:
    c = simulate_chain(P_split, 50_000, np.random.default_rng(2), start=start)
    print(f"  {start} 에서 출발: {c / c.sum()}")

# 아래는 눈으로만 확인하는 시연입니다. 정상분포가 하나가 아니라 한 가족입니다.
print("\n(시연) 아래 셋이 전부 pi @ P == pi 를 만족합니다")
for w in [1.0, 0.5, 0.0]:
    pi_w = np.array([w/2, w/2, (1-w)/2, (1-w)/2])
    print(f"  w = {w:.1f} -> {pi_w}   {np.allclose(pi_w @ P_split, pi_w)}")


Every mixture of the two worlds satisfies `πP = π`, so there is a whole **family** of stationary distributions and
no single "long-run behaviour" to speak of. Monte Carlo does not average over the family; it reports whichever
world you started in, and the answer changes with the starting state. That is what non-uniqueness looks like in practice.

One caveat worth keeping straight: reducible does **not** automatically mean non-unique. What creates the family here
is that both classes are *closed*. A chain with one closed class plus some transient states still has exactly one π,
the transient states simply get probability 0. You will see precisely that in the next section.

---
## 8. PageRank, the theorem shipped as a product

Four pages. A links to B and C; B links to C; C links back to A; D links to C and **nobody links to D**.
A random surfer clicks a uniformly chosen outgoing link. That is a Markov chain on pages, and PageRank is its
stationary distribution (deck: “PageRank, the Web's Stationary Distribution”).

In [ ]:
PAGES = ["A", "B", "C", "D"]

# 링크.  A -> B, C   ·   B -> C   ·   C -> A   ·   D -> C
# D 로 들어오는 링크는 없습니다.
# 각 행은 그 페이지에서 나가는 링크에 확률을 똑같이 나눠 준 것입니다.
M = np.array([[0.0, 0.5, 0.5, 0.0],    # A 에서 B 와 C 로 반반
              [0.0, 0.0, 1.0, 0.0],    # B 에서 C 로
              [1.0, 0.0, 0.0, 0.0],    # C 에서 A 로
              [0.0, 0.0, 1.0, 0.0]])   # D 에서 C 로
print("click matrix M (rows sum to 1):", M.sum(axis=1))

# 순간이동이 없으면 D 는 아무도 안 가리키므로 일시 상태가 됩니다.
print("\nM^60 (no teleport), first row:", np.linalg.matrix_power(M, 60)[0])

# 순간이동을 넣습니다. 확률 1-d 로 아무 페이지나 균등하게 골라 건너뜁니다.
d = 0.85
G = d * M + (1 - d) / 4 * np.ones((4, 4))
pr = stationary_by_solve(G)          # §3 에서 만든 함수를 그대로 씁니다

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(PAGES, pr, color="#3b6ea5", width=0.55)
ax.set_ylabel("PageRank"); ax.set_title("PageRank with teleportation (d = 0.85)")
plt.tight_layout(); plt.show()

for p, v in sorted(zip(PAGES, pr), key=lambda t: -t[1]):
    print(f"  {p}: {v:.4f}")
print(f"\nD gets exactly (1 - d)/4 = {(1 - d) / 4:.4f} - teleport traffic and nothing else")


Two things to take away.

- **Without teleportation** the chain is reducible: nobody links to D, so D is transient and `M^60` gives it
  probability 0, yet π is still unique, because {A, B, C} is the only closed class. This is the caveat from §7,
  made concrete.
- **With teleportation** every page reaches every other page in one hop, so the chain is irreducible and aperiodic
  by construction and the limit theorem applies. D's score is exactly `(1−d)/4 = 0.0375`: the floor that teleporting
  alone buys a page with no incoming links.

C wins (0.394) narrowly over A (0.373): C collects a link from all three other pages, but A is the only page C
points to, so A recycles most of C's score straight back. Importance is a fixed point, not a link count.

---
## 9. Your turn

Small changes, real answers. Do at least two.

1. Change the weather chain so Rainy is stickier (`P[2,2] = 0.8`, spreading the rest), re-solve, and say in one
   sentence which way π moved and why.
2. Add a fourth state "Snow" that can only be entered from Rainy and always goes back to Rainy next day.
   Is the chain still irreducible? Aperiodic? Check your answer against `P^n`.
3. Make an absorbing chain (the deck's {Playing, Won, Lost} from “Absorbing Chains, a Worked Miniature”) and use Monte Carlo to estimate the expected number
   of steps before absorption. The slide claims 5; do you get it?
4. In the PageRank example, add a link D ← A and recompute. Does D overtake B? Explain the ranking in one sentence.

Write one or two sentences under each thing you tried. The sentences are the point, not the numbers.

### Connect back to last week
In one sentence each, explain:
* Why counting sunny visits is a time average.
* Why a fixed transition table does not make a fixed Sunny start stationary.
* Why the alternator has a stable visit fraction without converging state probabilities.

> 요약: 계산 결과를 정상성·시간평균·상태확률 수렴의 구분으로 설명해 보세요.

Reading: Ross 11e §4.4 (Theorem 4.1) and §4.4.1; Week 2 deck “Weak Stationarity”, “Ergodicity” and “Three Things to Remember”.

In [ ]:
# Scratch space for section 9.


---
## 10. Environment record

Nothing to submit this week, but from Assignment 1 onward every submission ends with this cell; it is how a marker
reproduces your numbers. Run it once now so it is not new later.

In [ ]:
import sys, platform, datetime
try:
    import torch; gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"
except Exception:
    gpu = "torch not loaded"

print("run at      :", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("python      :", sys.version.split()[0], "on", platform.system())
print("runtime     :", gpu)
print("seed        :", SEED)
print("numpy       :", np.__version__)